The notebook shows how to use our tokenizer to encoder a point cloud to latent,
and then decode the latent to 3dgs.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import os
import typing as T

import numpy as np
import open3d as o3d

import torch

from lito.trainers import lito_trainer
from plibs import data_utils, gs_utils, o3d_utils, sh_utils, structures, utils

# assume you run the notebook in the notebooks folder
repo_root = os.path.abspath("..")

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    raise NotImplementedError("only support GPU currently")

In [ ]:
# we assume the tokenizer checkpoint is already downloaded.
tokenizer_checkpoint_filename = "https://ml-site.cdn-apple.com/models/lito/lito_new.ckpt"  # download it, or can also be a local path if already downloaded

from lito.eval_scripts.st_model_utils import download_checkpoint, is_http_url

if is_http_url(tokenizer_checkpoint_filename):
    tokenizer_checkpoint_filename = download_checkpoint(
        url=tokenizer_checkpoint_filename,
        download_dir_root=os.path.join(repo_root, "artifacts"),
        overwrite=False,
    )

model: lito_trainer.LightTokenizationTrainer = lito_trainer.LightTokenizationTrainer.load_from_checkpoint(
    checkpoint_path=tokenizer_checkpoint_filename,
    map_location=device,
    strict=False,  # we might not have lpips
)
model.freeze()
model.eval()

In [ ]:
# load data (see `render_data.ipynb` to see how the data was created)
data_filename = "assets/bunny.npz"
pdict = dict(np.load(data_filename, allow_pickle=True))
point_xyz_w = torch.from_numpy(pdict["point_xyz_w"]).to(device=device, dtype=torch.float)  # (n, 3) [-1, 1]
point_rgb = torch.from_numpy(pdict["point_rgb"]).to(device=device, dtype=torch.float)  # (n, 3) uint8
point_view_dir = torch.from_numpy(pdict["point_view_dir"]).to(
    device=device, dtype=torch.float
)  # (n, 3) uint8, from pinhole to point, unnormalized


# rgb and view_dir are saved as uint8 in the npz
point_rgb = point_rgb / 255  # (n, 3) [0, 1]
point_view_dir = torch.nn.functional.normalize(
    point_view_dir * (2 / 255) - 1,
    dim=-1,
)  # (n, 3), from pinhole to point

In [ ]:
# encode point cloud to latent
num_points = int(2**20)  # int(2 ** 18)  # the model is robust to fewer / more points
num_latent = 8192  # 4096  # 16384  # the model is robust to slightly fewer or more latents

with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=True):
    latent = model.get_latents(
        xyz_w=point_xyz_w[None, :num_points],  # (b=1, n, 3xyz_w)
        rgb=point_rgb[None, :num_points],  # (b=1, n, 3rgb)  [0, 1]
        ray_origin_direction_w=torch.cat(
            [
                point_xyz_w[None, :num_points],
                point_view_dir[None, :num_points],
            ],
            dim=-1,
        ),  # (b=1, n, 6)
        num_latent=num_latent,
    )["latent_tokens"]  # (b, num_latent, dim_latent)
    print(f"latent: {latent.shape}")

In [ ]:
# decode to 3dgs
result_dir = os.path.abspath("recon_results")
with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=True):
    print(f"decoding gaussian..")
    gs_dicts = model.inference_estimate_gaussians(
        fpoint_latent=latent,  # (b, num_latent, dim_latent)
        init_coord_src="voxel_decoder",
    )  # list of (b=1,)
    print(f"Finished")

    # save to ply
    print(f"saving gaussian..")
    filename = os.path.join(result_dir, f"reconstructed_bunny_3dgs_nl{num_latent}_np{num_points}.ply")
    ib = 0
    _sh_degree = sh_utils.get_sh_degree_from_total_dim(gs_dicts[ib]["rgb_sh"].size(-2))
    ngs = math.prod(gs_dicts[ib]["xyz_w"].shape[:-1])
    gs = gs_utils.Gaussians(
        sh_degree=_sh_degree,
        xyz_w=gs_dicts[ib]["xyz_w"].reshape(ngs, 3),  # (n, 3xyz)
        rgb_sh=gs_dicts[ib]["rgb_sh"].reshape(ngs, -1, 3),
        rgb_sh_dc=None,
        rgb_sh_rest=None,
        scaling_logit=None,
        quaternion_prenorm=None,
        opacity_logit=None,
        scaling=gs_dicts[ib]["scaling"].reshape(ngs, 3),  # (n, 3xyz)
        quaternion=gs_dicts[ib]["quaternion"].reshape(ngs, 4),  # (n, 4xyzw)
        opacity=gs_dicts[ib]["opacity"].reshape(ngs, 1),  # (n, 1)
        min_scaling=0,  # handled by network
        scaling_activation_type="none",
    )
    gs.save_ply(filename=filename)
    print(
        f"saved 3dgs to {filename}, can use supersplat https://superspl.at/editor or sparkjs https://sparkjs.dev/viewer/ to view"
    )

In [ ]:
# decode mesh
with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=True):
    print(f"decoding mesh..")
    raw_meshes: T.List[structures.RawMesh] = model.inference_estimate_mesh(
        fpoint_latent=latent,  # (b, num_latent, dim_latent)
        init_coord_src="voxel_decoder",
    )  # list of (b=1,)
    print(f"finished..")

    # save to ply
    print(f"saving mesh..")
    ib = 0
    o3d_mesh = raw_meshes[ib].get_o3d_mesh()
    filename = os.path.join(result_dir, f"reconstructed_bunny_mesh_nl{num_latent}_np{num_points}.ply")
    o3d.io.write_triangle_mesh(filename, o3d_mesh)
    print(f"saved mesh to {filename}")

In [ ]:
# sample point cloud
with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=True):
    print(f"sampling points..")
    num_points_to_sample = 16384
    init_noise_dict = model.get_conditional_sampling_init_noise(1, num_points_to_sample)
    sampled_x_flow_dict = model.conditional_sampling(
        fpoint_latent=latent,  # (b, num_latent, dim_latent)
        num_steps=100,
        **init_noise_dict,
        method="heun",
        latent_coord=None,  # (bl, dn) packed or None
    )  # (b, num_points_to_sample, d)
    sampled_xyz_w = sampled_x_flow_dict["xyz_w"].float()  # (b, num_points_to_sample, 3xyz_w)

    # save to ply
    print(f"saving points..")
    ib = 0
    o3d_pcd = o3d_utils.creat_pcd(
        points=sampled_xyz_w[ib],
        color=((init_noise_dict["init_xyz_w"][ib] + 1) * 0.5),  # use init uvw as color
    )
    filename = os.path.join(result_dir, f"reconstructed_bunny_pcd_nl{num_latent}_np{num_points}.ply")
    o3d.io.write_point_cloud(filename, o3d_pcd)
    print(f"saved points to {filename}")